In [1]:

from functools import partial
from toolz import memoize, pipe, accumulate,groupby, compose,compose_left, merge
from collections import namedtuple
import pathlib,os
import typing as T
import pydrake
from pydrake.all import (
    MultibodyPlant,
    ModelInstanceIndex,
    Body, namedview,
    JacobianWrtVariable,
    Frame,
)


import sympy as sp
import symforce
import symforce.symbolic as sf
import casadi as ca
from projects.refactor_mpb.symforce_casadi.casadi_config import CasadiConfig
from symforce.codegen.backends.pytorch import PyTorchConfig

import pydrake
from pydrake.symbolic import to_sympy as drake_to_sympy
from pydrake.all import (
    MultibodyPlant, MakeVectorVariable, JacobianWrtVariable, Frame,Body
)

from utils.my_sympy.misc import *
from utils.my_sympy.conversions import *
from utils.misc import *
from utils.my_drake.misc import *
from utils.math.quaternions import quaternion_dot_to_angular_velocity, hamiltonian_product
from utils.my_casadi.math import analytical_jacobian_to_geometric_matrix, quaternion_to_euler_angles, rot_matrix_to_quaternion


In [2]:
def drake_matrix_to_sympy(matrix, memo):
    matrix = matrix.tolist()
    row_list = []
    for row in matrix:
        column_list = []
        for element in row:
            element = pydrake.symbolic.to_sympy(element,memo = memo)
            column_list.append(element)
        row_list.append(column_list)
    return sp.Matrix(row_list)
def quaternion_exponential(v:Iterable,eps:float = 1e-8) -> ca.MX:
    # https://theorangeduck.com/page/exponential-map-angle-axis-angular-velocity
    half_angle = ca.norm_2(v)
    result = ca.if_else(half_angle < eps,ca.vertcat(1,v[0],v[1],v[2]),ca.vertcat(ca.cos(half_angle),*(v*ca.sin(half_angle)/half_angle).nz))
    return result

class CasadiMultiBodyPlantWrapper:
    """
    TODO: 
    - Dynamics
    - if using BSpline derivative, the derivatives of the orientation of free bodies are quaternion derivatives, not angular velocities
        - do something using the q_dot_to_angular_velocity function
    - maybe inherit from multibodyplant instead of having it as a member
    - FIX: having to call function(casadi_variable.nz) instead of function(casadi_variable)
    - calc_euler_angles_for_free_body(q,model_instance,'xyz')
    - conversion between jacobian types pg 24
        - https://ethz.ch/content/dam/ethz/special-interest/mavt/robotics-n-intelligent-systems/rsl-dam/documents/RobotDynamics2016/RD2016script.pdf
        - https://math.stackexchange.com/questions/3887845/mapping-from-quaternion-jacobian-to-geometric-jacobian
            quaternion one might me incorrect
    """
# q = ca.SX.sym('x',21,1)
# Js_1 = MPC.casadi_plant.calc_spatial_velocity_jacobian(q.nz,JacobianWrtVariable.kQDot, MPC.EE_frame_1,[0,0,0],MPC.world_frame,MPC.world_frame,clean_small_coeffs=1e-6)
# T = MPC.casadi_plant.calc_frame_pose_in_frame(q.nz,MPC.EE_frame_1,MPC.world_frame,clean_small_coeffs=1e-2)
# R = T[0:3,0:3]
# t = T[0:3,3]
# Js_c = ca.jacobian(R,q)
# Jt_c = ca.jacobian(t,q)


# # 
# # print(F1(q))
# # print(ca.densify(F2(q)))

# q_dot = ca.SX.sym('dx',21,1)
# W = (Js_c@q_dot).reshape((3,3))@R.T
# w_1 = W[2,1]
# w_2 = W[0,2]
# w_3 = W[1,0]
# F = ca.Function('F',[q,q_dot],[ca.cse(ca.vertcat(w_1,w_2,w_3,Jt_c@q_dot))])
# F2 = ca.Function('F2',[q,q_dot],[ca.cse(Js_1@q_dot)])
    def __init__(self, plant: MultibodyPlant,cse=True, temp_folder = None) -> None:
        self.plant = plant
        if temp_folder is None:
            self.TEMP_FOLDER = pathlib.Path(os.getcwd()) / 'temp'
        else:
            self.TEMP_FOLDER = pathlib.Path(temp_folder)
        self.TEMP_FOLDER.mkdir(exist_ok=True)
        
        self.mapping = {'ImmutableDenseMatrix': ca.blockcat,
                        'MutableDenseMatrix': ca.blockcat,
                        'Abs': ca.fabs
                        }
        
        self.cse = cse
        
        self.sym_plant = plant.ToSymbolic()
        self.sym_context = self.sym_plant.CreateDefaultContext()
        self.sym_world_frame = self.sym_plant.world_frame()
        self.drake_position_variables = MakeVectorVariable(
            self.plant.num_positions(), name='x')
        self.drake_velocity_variables = MakeVectorVariable(
            self.plant.num_velocities(), name='dx')
        self.sym_plant.SetPositions(self.sym_context, self.drake_position_variables)
        self.sym_plant.SetVelocities(self.sym_context, self.drake_velocity_variables)
        
        self.pos_vel_memo = merge({q.get_id(): sp.Symbol(f'q_{i}') for i,q in enumerate(self.drake_position_variables)}, {v.get_id(): sp.Symbol(f'v_{i}') for i,v in enumerate(self.drake_velocity_variables)})
        
        self.sympy_to_symforce = {v: sf.Symbol(v.name)  for k, v in self.pos_vel_memo.items()}

        self.sympy_velocity_variables = [drake_to_sympy(var, memo = self.pos_vel_memo) for var in self.drake_velocity_variables]
        self.sympy_position_variables = [drake_to_sympy(var, memo = self.pos_vel_memo) for var in self.drake_position_variables]
        
        self.free_bodies = get_all_free_bodies(self.plant)
        
        self.forward_euler = self.make_forward_integration_euler_function()
        self.plant_named_view = make_namedview_positions(plant,'')
        self.position_upper_limits = self.plant.GetPositionUpperLimits()
        self.position_lower_limits = self.plant.GetPositionLowerLimits()
    
    def get_free_body_position_with_euler_angles(self, q:Iterable, model_instance:ModelInstanceIndex, order:str):
        model_instances_of_free_bodies = [body.model_instance() for body in self.free_bodies]
        assert model_instance in model_instances_of_free_bodies, f"Model instance {model_instance} not in {model_instances_of_free_bodies}"
        current_position_object = self.get_positions_from_array(model_instance,q)
        current_euler_angles = quaternion_to_euler_angles(current_position_object[0:4],order,extrinsic=False)
        current_translation = current_position_object[4:]
        current_position_object = ca.vertcat(current_euler_angles,current_translation)
        return current_position_object
    def named_vector(self, name:str,vector:Iterable):
        """
        Gets a named view of the vector `vector` with name `name`.
        """
        if isinstance(vector,(ca.MX,ca.SX,ca.DM)):
            return self.plant_named_view(name,vector.nz)
        return self.plant_named_view(name,vector)
    def set_position_upper_limits(self, position_upper_limits:Iterable):
        self.position_upper_limits = position_upper_limits
    def set_position_lower_limits(self, position_lower_limits:Iterable):
        self.position_lower_limits = position_lower_limits
        
    def get_joints_midpoints(self):
        lower_limit = self.position_upper_limits
        upper_limit = self.position_lower_limits
        if isinstance(lower_limit, (tuple,list)):
            lower_limit = np.array(lower_limit)
        if isinstance(upper_limit, (tuple,list)):
            upper_limit = np.array(upper_limit)    
        q_middle = (lower_limit+upper_limit)/2
        return q_middle
    
    def get_positions_from_array(self, model_instance:ModelInstanceIndex, q:Iterable) -> Iterable:
        """
        Extract elements from the array q at indices that align with the generalized positions of `model_instance` within the whole plant position vector." 
        
        I.e: `return q[appropriate_indices]`
        
        Generic version of `plant.GetPositionsFromArray(model_instance,q)` that works with CasADi variables.
        """
        
        x = list(range(0,self.num_positions()))
        return q[self.plant.GetPositionsFromArray(model_instance,x).astype(int)]
    def set_positions_in_array(self, model_instance:ModelInstanceIndex, q:Iterable, q_instance:Iterable) -> Iterable:
        """
        Updates the positions in the array `q` based on the positions associated with `model_instance`. 
        
        This method takes indices corresponding to `model_instance` within the plant, and replaces 
        the values at these indices in `q` with the values from `q_instance`.
        
        I.e: `q[appropriate_indices] = q_instance`
        
        Generic version of `plant.SetPositionsInArray(model_instance,q_instance,q)` that works with CasADi variables.
        """
        x = list(range(0,self.num_positions()))
        q[self.plant.GetPositionsFromArray(model_instance,x).astype(int)] = q_instance
        return q
    
    def get_velocities_from_array(self, model_instance:ModelInstanceIndex, q:Iterable) -> Iterable:
        """
        Extract elements from the array q at indices that align with the velocities of `model_instance` within the whole plant velocity vector." 
        
        I.e: `return v[appropriate_indices]`
        
        Generic version of `plant.GetVelocitiesFromArray(model_instance,q)` that works with CasADi variables.
        
        The difference between this method and `get_positions_from_array` is that this method returns the velocities of the plant, 
        which has angular velocity instead of quaternion derivatives.
        """
        
        x = list(range(0,self.num_velocities()))
        return q[self.plant.GetVelocitiesFromArray(model_instance,x).astype(int)]
    def set_velocities_in_array(self, model_instance:ModelInstanceIndex, v:Iterable, v_instance:Iterable) -> Iterable:
        """
        Updates the positions in the array `v` based on the velocities associated with `model_instance`. 
        
        This method takes indices corresponding to `model_instance` within the plant, and replaces 
        the values at these indices in `v` with the values from `v_instance`.
        
        I.e: `v[appropriate_indices] = v_instance`
        
        Generic version of `plant.SetVelocitiesInArray(model_instance,q_instance,q)` that works with CasADi variables.
        """
        x = list(range(0,self.num_velocities()))
        v[self.plant.GetVelocitiesFromArray(model_instance,x).astype(int)] = v_instance
        return v
    def make_forward_integration_euler_function(self):
        """
        Makes a function that integrates the plant forward in time using Euler integration. Appropriately handles quaternions by using the exponential map.
        """
        q = ca.SX.sym('q',self.num_positions())
        v = ca.SX.sym('v',self.num_velocities())
        dt = ca.SX.sym('dt')
        
        
        v_named = namedview('',self.plant.GetVelocityNames())([a for a in v.nz])
        q_named = namedview('',self.plant.GetPositionNames())([a for a in q.nz])
        q_new = []
        for name in self.plant.GetPositionNames():
            last_el = name.split('_')[-1]
            name = name[:-len(last_el)-1]
            if last_el == 'qw':
                # v = 
                qw = eval("q_named" + "." +  name+ "_qw")
                qx = eval("q_named" + "." +  name+ "_qx")
                qy = eval("q_named" + "." +  name+ "_qy")
                qz = eval("q_named" + "." +  name+ "_qz")
                wx = eval("v_named" + "." +  name+ "_wx")
                wy = eval("v_named" + "." +  name+ "_wy")
                wz = eval("v_named" + "." +  name+ "_wz")
                
                quat = ca.vertcat(qw,qx,qy,qz)
                w = ca.vertcat(wx,wy,wz)
                new_quaternions = hamiltonian_product(quat,quaternion_exponential(w/2*dt))
                q_new.append(new_quaternions)
                pass
            elif last_el in ['qx','qy','qz']:
                
                continue
            else:
                
                state = eval("q_named" + "." +  name + "_" + last_el)
                def get_v_name(last_el):
                    match last_el:
                        case 'q':
                            return 'w'
                        case 'x':
                            return 'vx'
                        case 'y': 
                            return 'vy'
                        case 'z':
                            return 'vz'
                v_name = get_v_name(last_el)
                v_ = eval("v_named" + "." +  name + "_" + v_name)
                q_new.append(state + v_*dt)
        
        
        
        f = ca.Function('forward_integration_euler',[q,v,dt],[ca.vertcat(*q_new)],{'cse':True})
        return f
    
    @memoize
    @staticmethod
    def get_euler_dot_to_angular_velocity_matrix(order):
        psi, theta, phi = ca.SX.sym('psi'), ca.SX.sym('theta'), ca.SX.sym('phi')
        M = analytical_jacobian_to_geometric_matrix(order,[psi,theta,phi])
        
        return ca.Function('euler_dot_to_angular_velocity_matrix',[psi,theta,phi],[M],{'cse':True,'post_expand':True,})
    
    def calc_euler_dot_to_angular_velocity_matrix(self, euler_angles:Iterable, order):
        psi, theta, phi = euler_angles[0], euler_angles[1], euler_angles[2]
        return CasadiMultiBodyPlantWrapper.get_euler_dot_to_angular_velocity_matrix(order)(psi,theta,phi)
    
    @memoize 
    @staticmethod
    def get_angular_velocity_to_euler_dot_matrix(order):
        psi, theta, phi = ca.SX.sym('psi'), ca.SX.sym('theta'), ca.SX.sym('phi')
        M = analytical_jacobian_to_geometric_matrix(order,[psi,theta,phi])
        
        return ca.Function('angular_velocity_to_euler_dot_matrix',[psi,theta,phi],[M],{'cse':True,'post_expand':True,})
    def calc_angular_velocity_to_euler_dot_matrix(self, euler_angles:Iterable, order):
        psi, theta, phi = euler_angles[0], euler_angles[1], euler_angles[2]
        return CasadiMultiBodyPlantWrapper.get_angular_velocity_to_euler_dot_matrix(order)(psi,theta,phi)
    
    
    @memoize
    def get_frame_relative_velocity_function(self, frame:Frame, frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs):
        sym_frame:Frame = self.sym_plant.GetFrameByName(frame.name(),frame.model_instance())
        sym_frame_measured_in = self.sym_plant.GetFrameByName(frame_measured_in.name(),frame_measured_in.model_instance())
        sym_frame_expressed_in = self.sym_plant.GetFrameByName(frame_expressed_in.name(),frame_expressed_in.model_instance())
        velocity = sym_frame.CalcSpatialVelocity(self.sym_context,sym_frame_measured_in,sym_frame_expressed_in)
        translational_velocity = velocity.translational().tolist()
        rotational_velocity = velocity.rotational().tolist()
        velocity = [pydrake.symbolic.to_sympy(element) for element in rotational_velocity + translational_velocity]
        velocity = sp.Matrix(velocity).reshape(6,1)
        if clean_small_coeffs > 0:
            small_numbers = set([e for e in velocity.atoms(sympy.Number) if abs(e) < clean_small_coeffs])
            d = {s: 0 for s in small_numbers}
            velocity = velocity.subs(d)
        f = sp.lambdify([self.sympy_position_variables,self.sympy_velocity_variables],velocity,modules=[self.mapping,ca],cse=self.cse)
        return f
    def calc_frame_relative_velocity(self, q :Iterable, v :Iterable,frame:Frame, frame_measured_in:Frame,frame_expressed_in:Frame, clean_small_coeffs:float = -1):
        """
            Calculate the velocity of `frame` relative to `frame_measured_in` expressed in `frame_expressed_in` using `q` as the generalized positions of the plant and `v` as the velocities of the plant.
            
            Note: If there are quaternion derivatives in `v` you have to convert them to angular velocities first using `quaternion_dot_to_angular_velocity` for now.
            
            If `clean_small_coeffs` is set to a positive number, coefficients of the resulting velocity smaller than that number will be set to zero, simplifying the expression.
        """
        
        return self.get_frame_relative_velocity_function(frame,frame_measured_in,frame_expressed_in,clean_small_coeffs)(q,v)
    
    def get_sym_frame(self, frame:Frame):
        return self.sym_plant.GetFrameByName(frame.name(),frame.model_instance())

    # @memoize
    # def get_frame_pose_in_frame_function(self, frame:Frame, frame_ref:Frame, clean_small_coeffs):
    #     sym_frame:Frame = self.get_sym_frame(frame)
    #     sym_frame_ref = self.get_sym_frame(frame_ref)
        
    #     pose = drake_matrix_to_sympy(sym_frame.CalcPose(self.sym_context,sym_frame_ref).GetAsMatrix4(),self.pos_vel_memo)
    #     if clean_small_coeffs > 0:
    #         small_numbers = set([e for e in pose.atoms(sp.Number) if abs(e) < clean_small_coeffs])
    #         d = {s: 0 for s in small_numbers}
    #         pose = pose.subs(d)
            
    #     sympy_code = sp.python(pose,)
    #     sympy_code = "from sympy import *\n" + sympy_code
    #     with open(str(self.TEMP_FOLDER / "sympy_.py"),"w") as file:
    #         file.write(sympy_code)
    #     f = lambdify([self.sympy_position_variables],pose,modules=[self.mapping,ca],cse=self.cse)
    #     return f
    
    @memoize
    def get_frame_pose_in_frame_sympy(self, frame:Frame, frame_ref:Frame, clean_small_coeffs):
        sym_frame:Frame = self.get_sym_frame(frame)
        sym_frame_ref = self.get_sym_frame(frame_ref)
        
        pose = drake_matrix_to_sympy(sym_frame.CalcPose(self.sym_context,sym_frame_ref).GetAsMatrix4(),self.pos_vel_memo)
        if clean_small_coeffs > 0:
            small_numbers = set([e for e in pose.atoms(sp.Number) if abs(e) < clean_small_coeffs])
            d = {s: 0 for s in small_numbers}
            pose = pose.subs(d)
            

        return pose
    # def get_sympy_expression_or_make(self, function_name, function_to_call:T.Callable, path: pathlib.Path = None):
    #     if path is None: path = self.TEMP_FOLDER
        
    #     if not (path / f"{function_name}_sympy.py").exists():
    #         sympy_expr = function_to_call()
    #         sympy_code = sp.python(sympy_expr,)
    #         sympy_code = "from sympy import *\n" + sympy_code
    #         with open(str(self.TEMP_FOLDER / f"{function_name}_sympy.py"),"w") as file:
    #             file.write(sympy_code)
    #     else:
    #         modul = symforce.codegen.codegen_util.load_generated_package(function_name, path / f"{function_name}_sympy.py")
    #         sympy_expr = modul.e
    #     return sympy_expr
    @memoize(key = lambda args,kwargs: (kwargs["function_name"],kwargs["module"]) if "function_name" in kwargs else (args[1],kwargs["module"]))
    def get_function_or_make(self, function_name, function_to_call:T.Callable, path: pathlib.Path = None, module = 'casadi'):
        if path is None: path = self.TEMP_FOLDER
        if module == 'sympy':
            if not (path / f"{function_name}_sympy.py").exists():
                sympy_expr = function_to_call()
                sympy_code = sp.python(sympy_expr,)
                sympy_code = "from sympy import *\n" + sympy_code
                with open(str(self.TEMP_FOLDER / f"{function_name}_sympy.py"),"w") as file:
                    file.write(sympy_code)
            else:
                modul = symforce.codegen.codegen_util.load_generated_package(function_name, path / f"{function_name}_sympy.py")
                sympy_expr = modul.e
            return sympy_expr
        if module == 'casadi':
            if not (path / f"{function_name}_casadi.py").exists():
                function = function_to_call()
                return function
            else:
                modul = symforce.codegen.codegen_util.load_generated_package(function_name, path / f"{function_name}_casadi.py")
                function = getattr(modul, function_name)
                return function
        if module == 'pytorch':
            if not (path / f"{function_name}_pytorch.py").exists():
                function = function_to_call()
                return function
            else:
                modul = symforce.codegen.codegen_util.load_generated_package(function_name, path / f"{function_name}_pytorch.py")
                function = getattr(modul, function_name)
                return function
    
    def sympy_expression_to_symforce(self, sympy_expr, sympy_to_symforce_dict):
        assert isinstance(sympy_expr,(sp.MutableSparseMatrix,sp.ImmutableSparseMatrix,sp.Matrix,sp.MutableDenseMatrix,sp.ImmutableMatrix,sp.ImmutableDenseMatrix)) , "im only really considering matrices right now so edit this to change to symforce"
        if isinstance(sympy_expr,(sp.MutableSparseMatrix,sp.ImmutableSparseMatrix,sp.Matrix,sp.MutableDenseMatrix,sp.ImmutableMatrix,sp.ImmutableDenseMatrix)):
            sympy_expr = sf.Matrix(sympy_expr.subs(merge(*sympy_to_symforce_dict)).tolist())
        return sympy_expr
    @memoize(key = lambda args,kwargs: kwargs["function_name"] if "function_name" in kwargs else args[1])
    def sympy_to_pytorch(self, function_name, sympy_expr, inputs, path: pathlib.Path = None):
        if path is None: path = self.TEMP_FOLDER
        assert isinstance(inputs, (list,tuple)), "inputs must be a list or tuple of variables"
        
        function_name = function_name + "_pytorch"
        sympy_to_symforce = []
        for var in inputs:
            symbols = var.free_symbols
            sympy_to_symforce.append({v: sf.Symbol(v.name)  for v in symbols})
            
        sf_expr = self.sympy_expression_to_symforce(sympy_expr,sympy_to_symforce)
        
        codegen_input = symforce.values.Values()
        pytorch_input = []
        for i,var in enumerate(inputs):
            if len(sympy_to_symforce[i].values()) == 1:
                codegen_input[f"_in{i}"] = list(sympy_to_symforce[i].values())[0]
            else:
                codegen_input[f"_in{i}"] = sf.Matrix(list(sympy_to_symforce[i].values()))
            pytorch_input.append(sf.Symbol(f"_in{i}",len(sympy_to_symforce[i].values())))
        codegen_output = symforce.values.Values(out = sf_expr)
        
        codegen_obj = symforce.codegen.Codegen(
            inputs=codegen_input,
            outputs=codegen_output,
            config=PyTorchConfig(),
            name=function_name,
            return_key="out",
        )

        # filename.f
        generated_data = codegen_obj.generate_function(str(path), skip_directory_nesting=True)
        gen_module = symforce.codegen.codegen_util.load_generated_package(
            function_name, generated_data.function_dir
        )
        function = getattr(gen_module, function_name)
        return function
    
    @memoize(key = lambda args,kwargs: kwargs["function_name"] if "function_name" in kwargs else args[1])
    def sympy_to_casadi(self, function_name, sympy_expr, inputs, path: pathlib.Path = None):
        if path is None: path = self.TEMP_FOLDER
        assert isinstance(inputs, (list,tuple)), "inputs must be a list or tuple of variables"
        
        function_name = function_name + "_casadi"
        sympy_to_symforce = []
        for var in inputs:
            symbols = var.free_symbols
            sympy_to_symforce.append({v: sf.Symbol(v.name)  for v in symbols})
            
        sf_expr = self.sympy_expression_to_symforce(sympy_expr,sympy_to_symforce)
        
        codegen_input = symforce.values.Values()
        casadi_input = []
        for i,var in enumerate(inputs):
            if len(sympy_to_symforce[i].values()) == 1:
                codegen_input[f"_in{i}"] = list(sympy_to_symforce[i].values())[0]
            else:
                codegen_input[f"_in{i}"] = sf.Matrix(list(sympy_to_symforce[i].values()))
            casadi_input.append(ca.SX.sym(f"_in{i}",len(sympy_to_symforce[i].values())))
        codegen_output = symforce.values.Values(out = sf_expr)
        
        codegen_obj = symforce.codegen.Codegen(
            inputs=codegen_input,
            outputs=codegen_output,
            config=CasadiConfig(),
            name=function_name,
            return_key="out",
        )

        # filename.f
        generated_data = codegen_obj.generate_function(str(path), skip_directory_nesting=True)
        gen_module = symforce.codegen.codegen_util.load_generated_package(
            function_name, generated_data.function_dir
        )
        function = getattr(gen_module, function_name)
        casadi_function = ca.Function(function_name,casadi_input,[function(*casadi_input)],{'cse':True})
        return casadi_function

    @memoize
    def get_frame_pose_in_frame_function(self, frame:Frame, frame_ref:Frame, clean_small_coeffs:float = -1, module = 'casadi'):
        assert module in ['casadi','pytorch','sympy','numpy'] , "module must be one of ['casadi','pytorch','sympy','numpy']"
        
        function_name = f"frame_pose_in_frame_{frame.name()}_{frame_ref.name()}"
        inputs = [sp.Matrix(self.sympy_position_variables)]
        
        # if sympy function not done call this
        sympy_frame_partial = partial(self.get_frame_pose_in_frame_sympy, frame, frame_ref, clean_small_coeffs)
        # if other functions not created, call this then their respective functions
        sympy_frame_get_partial = partial(self.get_function_or_make, function_name = function_name, function_to_call = sympy_frame_partial,module = 'sympy')
        
        if module == 'casadi':
            casadi_partial = partial(self.sympy_to_casadi, function_name = function_name, inputs = inputs)
            func = self.get_function_or_make(function_name, compose_left(sympy_frame_get_partial,lambda expr: casadi_partial(sympy_expr = expr)), module = 'casadi')
            return func
        if module == 'pytorch':
            pytorch_partial = partial(self.sympy_to_pytorch, function_name = function_name, inputs = inputs)
            func = self.get_function_or_make(function_name, compose_left(sympy_frame_get_partial,lambda expr: pytorch_partial(sympy_expr = expr)), module = 'pytorch')
            return func
        if module == 'sympy':
            return sympy_frame_get_partial()
        
        
    def calc_frame_pose_in_frame(self, q :Iterable,frame:Frame, frame_expressed:Frame, clean_small_coeffs:float = -1, module = 'casadi'):
        """
            Calculate the pose of `frame` expressed `frame_ref` using `q` as the generalized positions of the plant.
            
            This means that the point `p` in `frame` is expressed in `frame_ref` as `p_ref = pose @ p`.
            
            If `clean_small_coeffs` is set to a positive number, coefficients of the resulting pose smaller than that number will be set to zero, simplifying the expression. (recommended: 1e-6)
        """
        # assert module in ['casadi','pytorch','sympy','numpy'] , "module must be one of ['casadi','pytorch','sympy','numpy']"
        # if module == 'casadi':
        return self.get_frame_pose_in_frame_function(frame,frame_expressed,clean_small_coeffs, module = module)(q)
            
        # raise NotImplementedError()
        # return self.get_frame_pose_in_frame_function(frame,frame_expressed,clean_small_coeffs)(q)
    def calc_frame_pose_in_frame_euler(self, q :Iterable,frame:Frame, frame_expressed:Frame, order:str,clean_small_coeffs:float = -1):
        """
            Calculate the pose of `frame` expressed `frame_ref` using `q` as the generalized positions of the plant.
            
            Returns the pose as an array `[psi,theta,phi,x,y,z]` where `psi,theta,phi` are the euler angles of the orientation of `frame` expressed in `frame_ref` and `x,y,z` are the translation of `frame` expressed in `frame_ref`.
            
            If `clean_small_coeffs` is set to a positive number, coefficients of the resulting pose smaller than that number will be set to zero, simplifying the expression. (recommended: 1e-6)
        """
        
        matrix = self.get_frame_pose_in_frame_function(frame,frame_expressed,clean_small_coeffs)(q)
        translation = matrix[0:3,3]
        quaternion = rot_matrix_to_quaternion(matrix[0:3,0:3])
        euler = quaternion_to_euler_angles(quaternion,order,extrinsic=False)
        return ca.vertcat(euler,translation)
        
        
    @memoize
    def get_spatial_velocity_jacobian(self, with_respect_to:JacobianWrtVariable,frame_of_point:Frame,frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs):
        sym_frame:Frame = self.sym_plant.GetFrameByName(frame_of_point.name(),frame_of_point.model_instance())
        sym_frame_measured_in = self.sym_plant.GetFrameByName(frame_measured_in.name(),frame_measured_in.model_instance())
        sym_frame_expressed_in = self.sym_plant.GetFrameByName(frame_expressed_in.name(),frame_expressed_in.model_instance())
        point = MakeVectorVariable(3, name='point')
        point_sympy = [pydrake.symbolic.to_sympy(var) for var in point]
        jacobian = self.sym_plant.CalcJacobianSpatialVelocity(self.sym_context, with_respect_to, sym_frame,point,sym_frame_measured_in,sym_frame_expressed_in).reshape(-1).tolist()
        jacobian = [pydrake.symbolic.to_sympy(element) for element in jacobian]
        if with_respect_to == JacobianWrtVariable.kQDot:
            
            jacobian = sp.Matrix(jacobian).reshape(6, self.num_positions())
        else:
            jacobian = sp.Matrix(jacobian).reshape(6, self.num_velocities())
        if clean_small_coeffs > 0:
            small_numbers = set([e for e in jacobian.atoms(sympy.Number) if abs(e) < clean_small_coeffs])
            d = {s: 0 for s in small_numbers}
            jacobian = jacobian.subs(d)
        f = lambdify([self.sympy_position_variables,point_sympy],jacobian,modules=[self.mapping,ca],cse=self.cse)
        return f
    def calc_spatial_velocity_jacobian(self, q :Iterable,with_respect_to:JacobianWrtVariable,frame_of_point:Frame,point:Iterable,frame_measured_in:Frame,frame_expressed_in, clean_small_coeffs:float = -1):
        """
        Get `[delta_rotation,delta_translation] / delta_q` of point `point` in `frame_of_point` relative to `frame_measured_in` expressed in `frame_expressed_in` with respect to `with_respect_to` variables calculated at plant position `q`.
        
        - `with_respect_to`: `JacobianWrtVariable.kQDot` or `JacobianWrtVariable.kV`
            -    `JacobianWrtVariable.kQDot`: the jacobian will be calculated with respect to the generalized velocities of the plant (so if there is a orientation of a free body in the plant, the jacobian will be calculated with respect to the quaternion (`[w_x,w_y,w_z]_point = J * [delta_q_w,delta_q_x,delta_q_y,delta_q_z]_plant`))
            -    `JacobianWrtVariable.kV`: the jacobian will be calculated with respect to the generalized velocities of the plant (so if there is a orientation of a free body in the plant, the jacobian will be calculated with respect to the angles (`[w_x,w_y,w_z]_point = J * [w_x,w_y,w_z]_plant`))

        OBS/TODO: This can get pretty slow (especially when substituting small coefficients) so maybe its a good idea to use `ca.jacobian` instead.
        
        """
        return self.get_spatial_velocity_jacobian(with_respect_to,frame_of_point,frame_measured_in,frame_expressed_in,clean_small_coeffs)(q,point)
    def calc_squared_distance_between_points(self, q :Iterable,frame_1:Frame, point_1:Iterable, frame_2:Frame,point_2:Iterable, clean_small_coeffs:float = -1, jacobian = False) -> Union[ca.MX,Iterable]:

        """
        Calculate the distance between `point_1` in `frame_1` and `point_2` in `frame_2` using `q` as the generalized positions of the plant.
        
        If `clean_small_coeffs` is set to a positive number, coefficients of the resulting distance smaller than that number will be set to zero, simplifying the expression.
        
        If `jacobian` is set to `True`, the function will also return the jacobian of the distance with respect to `q`. Returns `tuple(distance,jacobian)`.
        """
        frame_1_in_2_pose = self.calc_frame_pose_in_frame(q,frame_1,frame_2,clean_small_coeffs=clean_small_coeffs)
        p1 = frame_1_in_2_pose[0:3,0:3] @ point_1 + frame_1_in_2_pose[0:3,3]
        p2 = point_2
        distance = ca.sumsqr(p1-p2)
        if jacobian:
            jacobian = ca.jacobian(distance,q)
            return distance,jacobian
        else:
            return distance
    def calc_norm_2_distance_between_points(self, q :Iterable,frame_1:Frame, point_1:Iterable, frame_2:Frame,point_2:Iterable, clean_small_coeffs:float = -1, jacobian = False) -> Union[ca.MX,Iterable]:

        """
        Calculate the distance between `point_1` in `frame_1` and `point_2` in `frame_2` using `q` as the generalized positions of the plant.
        
        If `clean_small_coeffs` is set to a positive number, coefficients of the resulting distance smaller than that number will be set to zero, simplifying the expression.
        
        If `jacobian` is set to `True`, the function will also return the jacobian of the distance with respect to `q`. Returns `tuple(distance,jacobian)`.
        """
        frame_1_in_2_pose = self.calc_frame_pose_in_frame(q.nz,frame_1,frame_2,clean_small_coeffs=clean_small_coeffs)
        p1 = frame_1_in_2_pose[0:3,0:3] @ point_1 + frame_1_in_2_pose[0:3,3]
        p2 = point_2
        distance = ca.norm_2(p1-p2)
        if jacobian:
            jacobian = ca.jacobian(distance,q)
            return distance,jacobian
        else:
            return distance
    def forward_integration_euler(self, q :Iterable, v :Iterable, dt:float) -> Iterable:
        """
        Integrates the plant forward in time using Euler integration. Appropriately handles quaternions by using the exponential map.
        """
        # assert len(q) == len(v)
        # this check doesn't work with casadi variables so comment out if you want to do symbolic integration
        # if len(q) != len(v):
        #     raise ValueError(f"Length of q ({len(q)}) and v ({len(v)}) must be the same (non generalized velocities not implemented)")
        return self.forward_euler(q,v,dt)
    
    def num_positions(self):
        return self.plant.num_positions()

    def num_velocities(self):
         return self.plant.num_velocities()

In [3]:
pose.atoms

NameError: name 'pose' is not defined

In [ ]:
def f(x):
    return x
partial(f, x=2)()

2

In [3]:
from pydrake.all import (DiagramBuilder,AddMultibodyPlantSceneGraph,Parser,FindResourceOrThrow)
builder = DiagramBuilder()

plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.001)

    
franka_robot_urdf = FindResourceOrThrow("drake/manipulation/models/franka_description/urdf/panda_arm.urdf")
parser = Parser(plant)
(robot_1,) = Parser(plant).AddModels(franka_robot_urdf)
plant.RenameModelInstance(robot_1,"robot_1")
world_frame = plant.world_frame()

plant.WeldFrames(world_frame, plant.GetFrameByName('panda_link0',model_instance=robot_1), RigidTransform())

plant.Finalize()
casadi_plant = CasadiMultiBodyPlantWrapper(plant,cse = True)

{'fully_qualified_modules': False, 'inline': True, 'allow_unknown_functions': True, 'user_functions': {}}


In [4]:
plant_named_view = make_namedview_positions(plant,'')
context = plant.CreateDefaultContext()
plant.SetPositions(context, [1]*7)
# model instances
robot_1 = plant.GetModelInstanceByName("robot_1")
world_frame= plant.world_frame()
robot_1_link_7 = plant.GetFrameByName("panda_link7",robot_1)
robot_1_link_6 = plant.GetFrameByName("panda_link6",robot_1)

q = ca.SX.sym('dx',7,1)

f = casadi_plant.get_frame_pose_in_frame_function(robot_1_link_7,world_frame,1e-6)

casadi_plant.calc_frame_pose_in_frame(q,robot_1_link_7,world_frame,1e-6, module = 'casadi')
print(casadi_plant.calc_frame_pose_in_frame([1]*7,robot_1_link_7,world_frame,1e-6, module = 'casadi'))

print(robot_1_link_7.CalcPoseInWorld(context).GetAsMatrix4())

AttributeError: module 'frame_pose_in_frame_panda_link7_world' has no attribute 'frame_pose_in_frame_panda_link7_world'

In [43]:
f_torch = casadi_plant.get_frame_pose_in_frame_function(robot_1_link_7,world_frame,-1, module = 'pytorch')
import torch
print(f_torch(torch.tensor([1]*7)))

In [ ]:
from projects.refactor_mpb.symforce_casadi.casadi_config import CasadiConfig
import casadi as ca
from symforce.codegen import Codegen
import symforce.symbolic as sf
import typing as T
import symforce
from symforce.values import Values


def drake_matrix_to_sympy(matrix, memo):
    matrix = matrix.tolist()
    row_list = []
    for row in matrix:
        column_list = []
        for element in row:
            element = pydrake.symbolic.to_sympy(element,memo = memo)
            column_list.append(element)
        row_list.append(column_list)
    return sp.Matrix(row_list)
# T = casadi_plant.calc_frame_pose_in_frame(q.nz,robot_1_link_7,world_frame,clean_small_coeffs=1e-2)
sym_frame = casadi_plant.sym_plant.GetFrameByName(robot_1_link_7.name(),robot_1_link_7.model_instance())
sym_frame_ref = casadi_plant.sym_plant.GetFrameByName(world_frame.name(),world_frame.model_instance())
# pose = 
pos_vel_memo = {q.get_id(): sp.Symbol(f'q_{i}') for i,q in enumerate(casadi_plant.drake_position_variables)}
        
# pose = [pydrake.symbolic.to_sympy(element,memo = pos_vel_memo) for row in pose for element in row ]
pose = drake_matrix_to_sympy(sym_frame.CalcPose(casadi_plant.sym_context,sym_frame_ref).GetAsMatrix4(),pos_vel_memo)
sympy_to_symforce = {v: sf.Symbol(v.name)  for k, v in pos_vel_memo.items()}
sympy_vars = list(pos_vel_memo.values())
clean_pose = []
for p in pose:
    if isinstance(p,(float,int)):
        clean_pose.append(p)
        continue
    small_numbers = set([e for e in p.atoms(sp.Number) if abs(e) < 1e-6])
    d = {s: 0 for s in small_numbers}
    p = p.subs(d)
    clean_pose.append(p)
# pose = sp.Matrix(clean_pose).reshape(4,4)
# sympy_to_symforce = {v: sf.Symbol(v.name)  for k, v in self.pos_vel_memo.items()}
# 
# f = lambdify([sympy_vars],pose,modules=[casadi_plant.mapping,ca],cse=casadi_plant.cse)

In [ ]:
pose.free_symbols

{q_0, q_1, q_2, q_3, q_4, q_5, q_6}

In [ ]:
from projects.refactor_mpb.temp.sympy_ import e

In [ ]:
m = e.tolist()

In [ ]:


import casadi as ca
from projects.refactor_mpb.symforce_casadi.casadi_config import CasadiConfig
# from symforce.codegen.Codegen import Codegen
import symforce.symbolic as sf
from symforce.codegen.backends.pytorch import PyTorchConfig

import typing as T
import symforce
from symforce.values import Values

symforce.set_symbolic_api("symengine")
symforce.set_log_level("warning")
import pathlib,os
TEMP_FOLDER = pathlib.Path(os.getcwd()) / 'temp'
TEMP_FOLDER.mkdir(exist_ok=True)
inputs = symforce.values.Values()
pose = sf.Matrix(pose.subs(sympy_to_symforce).tolist())
inputs["q"] = sf.Matrix(list(sympy_to_symforce.values()))


display(inputs)
outputs = symforce.values.Values(ddang=pose)

display(outputs)
double_pendulum = symforce.codegen.Codegen(
    inputs=inputs,
    outputs=outputs,
    config=CasadiConfig(),
    name="double_pendulum",
    return_key="ddang",
)
# sf.
double_pendulum_data = double_pendulum.generate_function(str(TEMP_FOLDER), skip_directory_nesting=True)
double_pendulum = symforce.codegen.Codegen(
    inputs=inputs,
    outputs=outputs,
    config=PyTorchConfig(),
    name="double_pendulum_torch",
    return_key="ddang",
)
# sf.
double_pendulum_data = double_pendulum.generate_function(str(TEMP_FOLDER), skip_directory_nesting=True)

gen_module = symforce.codegen.codegen_util.load_generated_package(
    "t", double_pendulum_data.function_dir
)

Values(
  q: [q_0]
[q_1]
[q_2]
[q_3]
[q_4]
[q_5]
[q_6],
)

Values(
  ddang: [4.89658886014675e-12*sin(q_6)*(-1.0*sin(q_5)*(4.89658886014675e-12*sin(q_4)*(-1.0*sin(q_3)*(4.89658886014675e-12*sin(q_2)*(-4.89658886014675e-12*sin(q_0)*cos(q_1) - 1.0*sin(q_1)*cos(q_0)) - 1.0*sin(q_2)*sin(q_0) + 1.0*cos(q_2)*(-4.89658886014675e-12*sin(q_1)*sin(q_0) + 1.0*cos(q_0)*cos(q_1))) + 4.89658886014675e-12*cos(q_3)*(-1.0*sin(q_0)*cos(q_2) - 1.0*sin(q_2)*(-4.89658886014675e-12*sin(q_1)*sin(q_0) + 1.0*cos(q_0)*cos(q_1)) + 4.89658886014675e-12*cos(q_2)*(-4.89658886014675e-12*sin(q_0)*cos(q_1) - 1.0*sin(q_1)*cos(q_0))) + 1.0*cos(q_3)*(4.89658886014675e-12*sin(q_0)*cos(q_1) + 1.0*sin(q_1)*cos(q_0) - 4.89658886014675e-12*sin(q_0))) - 1.0*sin(q_4)*(2.39765824653132e-23*sin(q_0)*cos(q_1) + 1.0*sin(q_0)*cos(q_2) + 4.89658886014675e-12*sin(q_1)*cos(q_0) + 1.0*sin(q_2)*(-4.89658886014675e-12*sin(q_1)*sin(q_0) + 1.0*cos(q_0)*cos(q_1)) - 4.89658886014675e-12*cos(q_2)*(-4.89658886014675e-12*sin(q_0)*cos(q_1) - 1.0*sin(q_1)*cos(q_0)) - 2.39765824653132e-23*sin(q_0)) + 1.0*c

codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html



codegen.__init__():141 WARNING -- 
    Generating code with epsilon set to 0 - This is dangerous!  You may get NaNs, Infs,
    or numerically unstable results from calling generated functions near singularities.

    In order to safely generate code, you should set epsilon to either a symbol
    (recommended) or a small numerical value like `sf.numeric_epsilon`.  You should do
    this before importing any other code from symforce, e.g. with

        import symforce
        symforce.set_epsilon_to_symbol()

    or

        import symforce
        symforce.set_epsilon_to_number()

    For more information on use of epsilon to prevent singularities, take a look at the
    Epsilon Tutorial: https://symforce.org/tutorials/epsilon_tutorial.html



In [ ]:
import torch
gen_module.double_pendulum_torch(torch.tensor([1,2,3,4,5,6,7.]))

tensor([[-0.5236,  0.6902,  0.4994,  0.0323],
        [ 0.8287,  0.5487,  0.1106,  0.3014],
        [-0.1977,  0.4718, -0.8593,  0.6240],
        [ 0.0000,  0.0000,  0.0000,  1.0000]])

In [ ]:

def double_pendulum(q, tensor_kwargs=None):
    # type: (T.Sequence[torch.Tensor], TensorKwargs) -> torch.Tensor
    """
    This function was autogenerated. Do not modify by hand.

    Args:
        q: list

    Outputs:
        ddang: Matrix44


    """
    import casadi
    # Total ops: 178

    # Input arrays

    # Intermediate terms (61)
    _tmp0 = casadi.sin(q[0])
    _tmp1 = casadi.sin(q[2])
    _tmp2 = 1.0 * _tmp1
    _tmp3 = _tmp0 * _tmp2
    _tmp4 = casadi.cos(q[1])
    _tmp5 = casadi.cos(q[0])
    _tmp6 = casadi.cos(q[2])
    _tmp7 = 1.0 * _tmp6
    _tmp8 = _tmp5 * _tmp7
    _tmp9 = -_tmp3 + _tmp4 * _tmp8
    _tmp10 = casadi.sin(q[3])
    _tmp11 = 1.0 * _tmp10
    _tmp12 = casadi.cos(q[3])
    _tmp13 = 1.0 * _tmp12
    _tmp14 = casadi.sin(q[1])
    _tmp15 = _tmp14 * _tmp5
    _tmp16 = -_tmp11 * _tmp9 + _tmp13 * _tmp15
    _tmp17 = casadi.sin(q[5])
    _tmp18 = 1.0 * _tmp17
    _tmp19 = casadi.sin(q[4])
    _tmp20 = _tmp0 * _tmp7
    _tmp21 = _tmp2 * _tmp5
    _tmp22 = 1.0 * _tmp20 + 1.0 * _tmp21 * _tmp4
    _tmp23 = _tmp11 * _tmp15 + _tmp13 * _tmp9
    _tmp24 = casadi.cos(q[4])
    _tmp25 = 1.0 * _tmp24
    _tmp26 = -_tmp19 * _tmp22 + _tmp23 * _tmp25
    _tmp27 = casadi.cos(q[5])
    _tmp28 = 1.0 * _tmp27
    _tmp29 = _tmp16 * _tmp18 + _tmp26 * _tmp28
    _tmp30 = 1.0 * casadi.cos(q[6])
    _tmp31 = 1.0 * _tmp19
    _tmp32 = _tmp22 * _tmp24 + _tmp23 * _tmp31
    _tmp33 = 1.0 * casadi.sin(q[6])
    _tmp34 = _tmp0 * _tmp14
    _tmp35 = _tmp20 * _tmp4 + _tmp21
    _tmp36 = 1.0 * _tmp11 * _tmp34 + 1.0 * _tmp13 * _tmp35
    _tmp37 = _tmp3 * _tmp4 - _tmp8
    _tmp38 = _tmp24 * _tmp36 - _tmp31 * _tmp37
    _tmp39 = -_tmp11 * _tmp35 + _tmp13 * _tmp34
    _tmp40 = _tmp18 * _tmp39 + _tmp28 * _tmp38
    _tmp41 = _tmp19 * _tmp36 + _tmp25 * _tmp37
    _tmp42 = _tmp10 * _tmp4
    _tmp43 = _tmp14 * _tmp7
    _tmp44 = -_tmp12 * _tmp43 + 1.0 * _tmp42
    _tmp45 = _tmp14 * _tmp2
    _tmp46 = _tmp19 * _tmp45 + _tmp25 * _tmp44
    _tmp47 = _tmp10 * _tmp43 + _tmp13 * _tmp4
    _tmp48 = _tmp18 * _tmp47 + _tmp28 * _tmp46
    _tmp49 = -_tmp24 * _tmp45 + _tmp31 * _tmp44
    _tmp50 = 0.384 * _tmp10
    _tmp51 = 0.0825 * _tmp12
    _tmp52 = 0.316 * _tmp14
    _tmp53 = 0.0825 * _tmp1
    _tmp54 = 0.088 * _tmp17
    _tmp55 = 0.088 * _tmp27
    _tmp56 = 0.0825 * _tmp6
    _tmp57 = _tmp4 * _tmp56
    _tmp58 = 0.384 * _tmp12
    _tmp59 = 0.0825 * _tmp10
    _tmp60 = _tmp14 * _tmp56

    # Output terms
    _ddang = casadi.blockcat(
        [
            [
                _tmp29 * _tmp30 + _tmp32 * _tmp33,
                -_tmp29 * _tmp33 + _tmp30 * _tmp32,
                -_tmp16 * _tmp28 + _tmp18 * _tmp26,
                -_tmp0 * _tmp53
                + _tmp15 * _tmp58
                - _tmp15 * _tmp59
                + _tmp16 * _tmp54
                + _tmp26 * _tmp55
                + _tmp5 * _tmp52
                + _tmp5 * _tmp57
                - _tmp50 * _tmp9
                - _tmp51 * _tmp9,
            ],
            [
                _tmp30 * _tmp40 + _tmp33 * _tmp41,
                _tmp30 * _tmp41 - _tmp33 * _tmp40,
                _tmp18 * _tmp38 - _tmp28 * _tmp39,
                _tmp0 * _tmp52
                + _tmp0 * _tmp57
                + _tmp34 * _tmp58
                - _tmp34 * _tmp59
                - _tmp35 * _tmp50
                - _tmp35 * _tmp51
                + _tmp38 * _tmp55
                + _tmp39 * _tmp54
                + _tmp5 * _tmp53,
            ],
            [
                _tmp30 * _tmp48 + _tmp33 * _tmp49,
                _tmp30 * _tmp49 - _tmp33 * _tmp48,
                _tmp18 * _tmp46 - _tmp28 * _tmp47,
                _tmp12 * _tmp60
                + _tmp14 * _tmp50 * _tmp6
                + _tmp4 * _tmp58
                + 0.316 * _tmp4
                - 0.0825 * _tmp42
                + _tmp46 * _tmp55
                + _tmp47 * _tmp54
                - _tmp60
                + 0.333,
            ],
            [0, 0, 0, 1.00000000000000],
        ]
    )

    return _ddang

q = ca.SX.sym('dx',7,1)
double_pendulum(q).shape

In [ ]:
casadi_plant.sympy_position_variables[0].name

In [ ]:
type(casadi_plant.sympy_position_variables[0])